In [ ]:
import pandas as pd
import numpy as np

# sep=None lets pandas detect whether the file uses ',' or ';'
df = pd.read_csv("student-por.csv", sep=None, engine="python")
print(df.shape)

# ---- Settings (match Member 2's work) ----
selected_features = ['age', 'studytime', 'failures', 'absences', 'schoolsup', 'famsup',
                     'Medu', 'Fedu', 'internet', 'higher', 'goout', 'health',
                     'traveltime', 'freetime']
PASS_MARK = 10   # Pass = final grade G3 >= 10

if 'Pass' not in df.columns:
    df['Pass'] = (df['G3'] >= PASS_MARK).astype(int)

In [ ]:
X = df[selected_features].copy()
y = df['Pass']

# encode yes/no columns as 1/0
for col in ['schoolsup', 'famsup', 'internet', 'higher']:
    X[col] = X[col].map({'yes': 1, 'no': 0})

# if Pass is stored as text, convert it
if y.dtype == object:
    y = y.map({'Pass': 1, 'Fail': 0})

print(X.dtypes)
print("Missing values in X:", X.isnull().sum().sum())
print(y.value_counts())
print(y.value_counts(normalize=True).round(3))

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print("Train:", X_train.shape, "Test:", X_test.shape)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

log_reg = make_pipeline(StandardScaler(),
                        LogisticRegression(max_iter=1000, class_weight='balanced'))
dt = DecisionTreeClassifier(random_state=42, class_weight='balanced')
rf = RandomForestClassifier(random_state=42, class_weight='balanced')

log_reg.fit(X_train, y_train)
dt.fit(X_train, y_train)
rf.fit(X_train, y_train)
print("Baseline models trained.")

In [ ]:
from sklearn.model_selection import GridSearchCV

lr_grid = GridSearchCV(
    log_reg,
    {"logisticregression__C": [0.01, 0.1, 1, 10, 100]},
    cv=5, scoring="f1")
lr_grid.fit(X_train, y_train)

In [ ]:
dt_grid = GridSearchCV(
    DecisionTreeClassifier(random_state=42, class_weight='balanced'),
    {"max_depth": [3, 5, 7, 10, None],
     "min_samples_split": [2, 5, 10],
     "min_samples_leaf": [1, 2, 4]},
    cv=5, scoring="f1")
dt_grid.fit(X_train, y_train)

In [ ]:
# This may take a minute or two
rf_grid = GridSearchCV(
    RandomForestClassifier(random_state=42, class_weight='balanced'),
    {"n_estimators": [100, 200, 300],
     "max_depth": [5, 10, None],
     "min_samples_split": [2, 5, 10]},
    cv=5, scoring="f1", n_jobs=-1)
rf_grid.fit(X_train, y_train)

In [ ]:
results = pd.DataFrame({
    "Model": ["Logistic Regression", "Decision Tree", "Random Forest"],
    "Best CV F1": [lr_grid.best_score_, dt_grid.best_score_, rf_grid.best_score_],
    "Best Parameters": [lr_grid.best_params_, dt_grid.best_params_, rf_grid.best_params_],
})
results["Best CV F1"] = results["Best CV F1"].round(3)
pd.set_option("display.max_colwidth", None)
results

In [ ]:
import joblib

joblib.dump(lr_grid.best_estimator_, "best_logreg.pkl")
joblib.dump(dt_grid.best_estimator_, "best_tree.pkl")
joblib.dump(rf_grid.best_estimator_, "best_rf.pkl")
joblib.dump((X_train, X_test, y_train, y_test), "split_data.pkl")
print("Saved: best_logreg.pkl, best_tree.pkl, best_rf.pkl, split_data.pkl")